# UTS Data Acquisition & Management — Direktori Akomodasi Ubud

**Nama:** Ni Putu Riska Pramesti Cahya Dewi  
**NIM:** 2501010315  
**Program Studi:** Informatika — INSTIKI

Notebook ini menggunakan dataset hasil ekstraksi dari publikasi *Direktori Hotel Kabupaten Gianyar 2017* (data tahun 2015). Tidak ada data hotel atau desa buatan yang digunakan. Data cuaca diambil dari Open-Meteo Historical Weather API untuk 1–31 Januari 2025. Karena periode kedua sumber berbeda, cuaca hanya menjadi konteks wilayah, bukan kondisi cuaca pada tahun 2015.

**Penting:** Jalankan seluruh notebook dari awal sampai akhir. Seluruh output dibuat ulang dari file sumber dan respons API aktual. Jika sumber tidak tersedia atau API gagal, notebook akan berhenti dengan pesan yang jelas; notebook tidak membuat data pengganti.


## 1. Persiapan folder dan pustaka

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, time, subprocess
import requests
import pandas as pd
from IPython.display import display

# Google Colab tidak otomatis mengunduh folder data saat notebook dibuka dari GitHub.
# Karena itu, clone repository terlebih dahulu jika file sumber belum tersedia.
REPO_URL = "https://github.com/niputuriskapramesticahyadewi-maker/UTS_DAM_UBUD_RISKA_2501010315.git"
REPO_DIR = Path("/content/UTS_DAM_UBUD_RISKA_2501010315")
NAMA_FOLDER = "UTS_DAM_UBUD_RISKA_2501010315_FINAL"

def csv_tersedia(folder):
    return (folder / "data" / "interim" / "akomodasi_ubud_bps_2015_ekstraksi.csv").is_file()

# Coba folder kerja yang mungkin sudah ada.
kandidat = [
    Path.cwd() / NAMA_FOLDER,
    Path.cwd(),
    REPO_DIR / NAMA_FOLDER,
    Path("/content") / NAMA_FOLDER,
]
BASE_DIR = next((p for p in kandidat if csv_tersedia(p)), None)

# Jika belum ada, unduh repository GitHub beserta folder data.
if BASE_DIR is None:
    if REPO_DIR.exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    BASE_DIR = REPO_DIR / NAMA_FOLDER

if not csv_tersedia(BASE_DIR):
    raise FileNotFoundError(
        "CSV sumber masih belum ditemukan. Pastikan repository GitHub berisi "
        "data/interim/akomodasi_ubud_bps_2015_ekstraksi.csv."
    )

RAW_DIR = BASE_DIR / "data" / "raw"
INTERIM_DIR = BASE_DIR / "data" / "interim"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
META_DIR = BASE_DIR / "metadata"
DOCS_DIR = BASE_DIR / "docs"
for folder in [RAW_DIR, INTERIM_DIR, PROCESSED_DIR, META_DIR, DOCS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

DATASET_PATH = INTERIM_DIR / "akomodasi_ubud_bps_2015_ekstraksi.csv"
PDF_PATH = RAW_DIR / "BPS5104-direktori_hotel_kabupaten_gianyar_2017.pdf"
if not DATASET_PATH.is_file():
    raise FileNotFoundError(f"CSV sumber tidak ditemukan: {DATASET_PATH}")
if not PDF_PATH.is_file():
    raise FileNotFoundError(f"PDF sumber tidak ditemukan: {PDF_PATH}")

def get_checked(url, params=None, timeout=30):
    response = requests.get(
        url, params=params, timeout=timeout,
        headers={"User-Agent": "UTS-DAM-Student-Project/1.0"}
    )
    response.raise_for_status()
    return response

print("Folder proyek:", BASE_DIR)
print("CSV sumber tersedia:", DATASET_PATH.exists())
print("PDF BPS tersedia:", PDF_PATH.exists())


## 2. Membaca dan memvalidasi dataset BPS

In [ ]:
df_akomodasi = pd.read_csv(DATASET_PATH, encoding="utf-8-sig")
kolom_wajib = ["no_sumber", "desa", "nama_akomodasi", "alamat", "klasifikasi", "kecamatan", "tahun_data"]
kolom_hilang = sorted(set(kolom_wajib) - set(df_akomodasi.columns))
if kolom_hilang:
    raise ValueError(f"Kolom wajib tidak ditemukan: {kolom_hilang}")
if len(df_akomodasi) < 100:
    raise ValueError(f"Dataset hanya memiliki {len(df_akomodasi)} baris; minimal 100.")
if df_akomodasi["nama_akomodasi"].isna().any():
    raise ValueError("Ada nama akomodasi kosong.")
if df_akomodasi.duplicated().any():
    raise ValueError("Ditemukan baris duplikat pada dataset sumber.")
if df_akomodasi["nama_akomodasi"].astype(str).str.contains(r"Hotel Ubud Indah|Desa_Ubud_", case=False, regex=True).any():
    raise ValueError("Terindikasi data dummy. Hentikan analisis dan periksa sumber CSV.")

print("Jumlah baris:", len(df_akomodasi))
print("Jumlah kolom:", len(df_akomodasi.columns))
print("Tahun data:", sorted(df_akomodasi["tahun_data"].dropna().unique().tolist()))
print("Desa/kelurahan:", df_akomodasi["desa"].nunique())
print("Duplikasi:", int(df_akomodasi.duplicated().sum()))
display(df_akomodasi.head())
display(df_akomodasi["klasifikasi"].value_counts().to_frame("jumlah"))


## 3. Analisis berdasarkan dataset sumber BPS

Analisis berikut memakai seluruh baris aktual pada CSV hasil ekstraksi publikasi BPS. Tidak dibuat nama hotel, desa, atau jumlah data contoh.

In [ ]:
# Analisis deskriptif dari data BPS yang sudah dibaca dan divalidasi.
# Semua jumlah dihitung langsung dari df_akomodasi, bukan angka yang diketik manual.

analisis_desa = (
    df_akomodasi.groupby("desa", dropna=False)
    .agg(jumlah_akomodasi=("nama_akomodasi", "count"))
    .sort_values("jumlah_akomodasi", ascending=False)
    .reset_index()
)

analisis_klasifikasi = (
    df_akomodasi.groupby("klasifikasi", dropna=False)
    .agg(jumlah_akomodasi=("nama_akomodasi", "count"))
    .sort_values("jumlah_akomodasi", ascending=False)
    .reset_index()
)

print("Jumlah baris dataset:", len(df_akomodasi))
print("Jumlah nama akomodasi unik:", df_akomodasi["nama_akomodasi"].nunique())
print("Jumlah desa/kelurahan:", df_akomodasi["desa"].nunique())
print("\nDistribusi akomodasi berdasarkan desa:")
display(analisis_desa)
print("\nDistribusi berdasarkan klasifikasi:")
display(analisis_klasifikasi)

# Simpan tabel analisis agar angka yang dilaporkan dapat ditelusuri.
analisis_desa.to_csv(INTERIM_DIR / "analisis_akomodasi_per_desa.csv", index=False, encoding="utf-8-sig")
analisis_klasifikasi.to_csv(INTERIM_DIR / "analisis_akomodasi_per_klasifikasi.csv", index=False, encoding="utf-8-sig")

## 5. Akuisisi artikel Wikipedia terkait Ubud melalui MediaWiki API

Tahap ini menggunakan API resmi Wikipedia bahasa Indonesia. Hasilnya adalah metadata dan teks artikel terkait Ubud, bukan direktori lengkap hotel. Data disimpan sebagai JSON mentah dan CSV turunan.

In [ ]:
WIKI_API = "https://id.wikipedia.org/w/api.php"
WIKI_RAW_PATH = RAW_DIR / "wikipedia_ubud_search_raw.json"
WIKI_CSV_PATH = INTERIM_DIR / "wikipedia_ubud_articles.csv"
wiki_keywords = ["Ubud", "hotel di Ubud", "penginapan Ubud", "akomodasi Ubud", "resor Ubud", "villa Ubud"]
wiki_records = []
wiki_raw_responses = []

wiki_headers = {"User-Agent": "UTS-DAM-Ubud-Student-Research/1.0 (academic project)"}
for keyword in wiki_keywords:
    params = {
        "action": "query", "list": "search", "srsearch": keyword,
        "srlimit": 20, "format": "json"
    }
    response = requests.get(WIKI_API, params=params, headers=wiki_headers, timeout=30)
    response.raise_for_status()
    payload = response.json()
    wiki_raw_responses.append({"keyword": keyword, "response": payload})

    for item in payload.get("query", {}).get("search", []):
        page_id = item.get("pageid")
        if page_id is None:
            continue
        detail_params = {
            "action": "query", "pageids": page_id,
            "prop": "extracts|info", "explaintext": 1,
            "inprop": "url", "format": "json"
        }
        detail_response = requests.get(
            WIKI_API, params=detail_params, headers=wiki_headers, timeout=30
        )
        detail_response.raise_for_status()
        detail_payload = detail_response.json()
        wiki_raw_responses.append({
            "keyword": keyword, "page_id": page_id, "response": detail_payload
        })
        page = detail_payload.get("query", {}).get("pages", {}).get(str(page_id), {})
        if page:
            wiki_records.append({
                "keyword_pencarian": keyword,
                "judul": page.get("title", ""),
                "page_id": page_id,
                "ringkasan": page.get("extract", ""),
                "url_sumber": page.get("fullurl", "")
            })
        time.sleep(1)

# Hapus artikel berulang berdasarkan page_id, bukan mengubah isi sumber.
df_wikipedia = pd.DataFrame(wiki_records)
if not df_wikipedia.empty:
    df_wikipedia = df_wikipedia.drop_duplicates(subset=["page_id"])

WIKI_RAW_PATH.write_text(
    json.dumps(wiki_raw_responses, ensure_ascii=False, indent=2),
    encoding="utf-8"
)
df_wikipedia.to_csv(WIKI_CSV_PATH, index=False, encoding="utf-8-sig")

print("Artikel Wikipedia unik:", len(df_wikipedia))
print("JSON mentah tersimpan:", WIKI_RAW_PATH)
print("CSV hasil ekstraksi tersimpan:", WIKI_CSV_PATH)
display(df_wikipedia.head(10))

## 6. Etika data dan checklist

In [ ]:
CHECKLIST_CSV = META_DIR / "data_ethics_checklist.csv"
if CHECKLIST_CSV.exists():
    df_checklist = pd.read_csv(CHECKLIST_CSV, encoding="utf-8-sig")
    display(df_checklist)
else:
    print("Checklist belum tersedia; buat dokumentasi checklist secara terpisah sebelum pengumpulan.")

print("Kolom yang berpotensi menjadi pengenal:", ["nama_akomodasi", "alamat", "desa"])
print("Kolom nomor telepon tersedia:", any("telepon" in str(c).lower() or "phone" in str(c).lower() for c in df_akomodasi.columns))
print("Catatan: data usaha/alamat bukan otomatis data pribadi; tetap batasi penggunaan pada tujuan akademik dan ikuti ketentuan sumber.");


## 7. Akuisisi data cuaca melalui API Open-Meteo

In [ ]:
API_URL = "https://archive-api.open-meteo.com/v1/archive"
parameter_api = {
    "latitude": -8.5069,
    "longitude": 115.2625,
    "start_date": "2025-01-01",
    "end_date": "2025-01-31",
    "daily": "temperature_2m_mean,precipitation_sum,wind_speed_10m_max",
    "timezone": "Asia/Makassar"
}
api_response = get_checked(API_URL, params=parameter_api)
raw_api = api_response.json()
if "daily" not in raw_api or "time" not in raw_api["daily"]:
    raise ValueError("Respons Open-Meteo tidak memiliki struktur daily/time yang diharapkan.")

daily = raw_api["daily"]
df_cuaca = pd.DataFrame({
    "tanggal": daily["time"],
    "suhu_rata2_c": daily["temperature_2m_mean"],
    "curah_hujan_mm": daily["precipitation_sum"],
    "kecepatan_angin_maks_kmh": daily["wind_speed_10m_max"]
})
if len(df_cuaca) < 30 or df_cuaca["tanggal"].duplicated().any():
    raise ValueError("Data cuaca tidak memenuhi minimal 30 tanggal unik.")

API_RAW_PATH = RAW_DIR / "open_meteo_ubud_2025-01.json"
API_RAW_PATH.write_text(json.dumps(raw_api, ensure_ascii=False, indent=2), encoding="utf-8")
CUACA_CSV_PATH = INTERIM_DIR / "cuaca_harian_ubud_2025-01.csv"
df_cuaca.to_csv(CUACA_CSV_PATH, index=False, encoding="utf-8-sig")

print("HTTP status:", api_response.status_code)
print("Jumlah observasi harian:", len(df_cuaca))
print("Periode:", df_cuaca["tanggal"].min(), "s.d.", df_cuaca["tanggal"].max())
display(df_cuaca.head())


## 8. Pemeriksaan robots.txt BPS

In [ ]:
from urllib.parse import urljoin
from urllib.robotparser import RobotFileParser

SCRAPE_BASE = "https://bali.bps.go.id"
ROBOTS_URL = urljoin(SCRAPE_BASE, "/robots.txt")
SCRAPE_URL = "https://bali.bps.go.id/"
robots_status = None
status_scraping = "BELUM DIPERIKSA"
robots_path = RAW_DIR / "robots_bali_bps.txt"

try:
    robots_response = requests.get(ROBOTS_URL, timeout=20, headers={"User-Agent":"UTS-DAM-Student-Project/1.0"})
    robots_status = robots_response.status_code
    if robots_response.ok:
        robots_path.write_text(robots_response.text, encoding="utf-8")
        parser = RobotFileParser()
        parser.parse(robots_response.text.splitlines())
        boleh = parser.can_fetch("UTS-DAM-Student-Project/1.0", SCRAPE_URL)
        status_scraping = "DIIZINKAN" if boleh else "TIDAK DIIZINKAN"
    else:
        status_scraping = f"TIDAK DILAKUKAN: robots.txt HTTP {robots_status}"
except requests.RequestException as exc:
    status_scraping = f"TIDAK DILAKUKAN: pemeriksaan robots.txt gagal ({type(exc).__name__}: {exc})"

bukti = (
    f"URL robots.txt: {ROBOTS_URL}\n"
    f"HTTP status: {robots_status}\n"
    f"Status: {status_scraping}\n"
    "Scraping tidak dilakukan jika robots.txt tidak dapat diperiksa atau akses tidak diizinkan.\n"
)
(DOCS_DIR / "bukti_scraping.txt").write_text(bukti, encoding="utf-8")
print(bukti)


## 9. Akuisisi data titik akomodasi melalui OpenStreetMap Overpass API

In [ ]:
# Akuisisi satu kali melalui Overpass API. Tidak melakukan request paralel.
# Batas koordinat adalah bounding box perkiraan area Ubud, bukan batas administratif resmi kecamatan.
OVERPASS_URL = "https://overpass-api.de/api/interpreter"
OSM_RAW_PATH = RAW_DIR / "openstreetmap_akomodasi_ubud_raw.json"
OSM_CSV_PATH = INTERIM_DIR / "openstreetmap_akomodasi_ubud.csv"
OSM_ERROR_PATH = RAW_DIR / "openstreetmap_overpass_error.txt"

south, west, north, east = -8.57, 115.20, -8.42, 115.34
tourism_values = "hotel|guest_house|hostel|apartment|chalet|motel|camp_site"
overpass_query = f"""
[out:json][timeout:60];
(
  node["tourism"~"^{tourism_values}$"]({south},{west},{north},{east});
  way["tourism"~"^{tourism_values}$"]({south},{west},{north},{east});
  relation["tourism"~"^{tourism_values}$"]({south},{west},{north},{east});
);
out center tags;
"""
osm_headers = {
    "User-Agent": "UTS-DAM-Ubud-Student-Research/1.0 (academic coursework)"
}

try:
    osm_response = requests.post(
        OVERPASS_URL,
        data={"data": overpass_query},
        headers=osm_headers,
        timeout=90
    )
    osm_response.raise_for_status()
    osm_raw = osm_response.json()
    if "elements" not in osm_raw:
        raise ValueError("Respons Overpass tidak memiliki daftar elements.")

    # Simpan respons mentah sebelum ekstraksi CSV.
    OSM_RAW_PATH.write_text(
        json.dumps(osm_raw, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )

    osm_records = []
    for element in osm_raw.get("elements", []):
        tags = element.get("tags", {})
        center = element.get("center", {})
        lat = element.get("lat", center.get("lat"))
        lon = element.get("lon", center.get("lon"))
        osm_type = element.get("type", "")
        osm_id = element.get("id")
        osm_records.append({
            "osm_type": osm_type,
            "osm_id": osm_id,
            "nama": tags.get("name"),
            "tourism": tags.get("tourism"),
            "alamat": tags.get("addr:street"),
            "desa_atau_suburb": tags.get("addr:suburb") or tags.get("addr:hamlet"),
            "kota": tags.get("addr:city"),
            "latitude": lat,
            "longitude": lon,
            "url_objek": f"https://www.openstreetmap.org/{osm_type}/{osm_id}"
        })

    df_osm = pd.DataFrame(osm_records)
    if not df_osm.empty:
        df_osm = df_osm.drop_duplicates(subset=["osm_type", "osm_id"])
    OSM_ERROR_PATH.unlink(missing_ok=True)
    print("HTTP status Overpass:", osm_response.status_code)
    print("Waktu basis data OSM:", osm_raw.get("osm3s", {}).get("timestamp_osm_base", "tidak tersedia"))

except (requests.RequestException, ValueError) as exc:
    # Jangan mengarang hasil. Catat kegagalan dan lanjutkan notebook dengan tabel kosong.
    df_osm = pd.DataFrame(columns=[
        "osm_type", "osm_id", "nama", "tourism", "alamat",
        "desa_atau_suburb", "kota", "latitude", "longitude", "url_objek"
    ])
    OSM_ERROR_PATH.write_text(
        f"Akuisisi Overpass gagal: {type(exc).__name__}: {exc}\n"
        "Tidak ada hasil OSM yang dibuat. Jalankan ulang nanti jika API tersedia.",
        encoding="utf-8"
    )
    print("Akuisisi Overpass gagal:", type(exc).__name__, str(exc))
    print("Kegagalan dicatat di:", OSM_ERROR_PATH)

df_osm.to_csv(OSM_CSV_PATH, index=False, encoding="utf-8-sig")
print("Jumlah objek akomodasi OpenStreetMap:", len(df_osm))
print("JSON mentah tersedia:", OSM_RAW_PATH.exists())
print("CSV hasil ekstraksi:", OSM_CSV_PATH)
print("Atribusi: © OpenStreetMap contributors — Open Database License (ODbL)")
display(df_osm.head(10))


## 10. Penggabungan data dan validasi

In [ ]:
# Ringkas cuaca harian menjadi satu baris untuk Kecamatan Ubud.
df_cuaca_ringkas = pd.DataFrame([{
    "kecamatan": "Ubud",
    "periode_cuaca": "2025-01",
    "suhu_rata2_bulanan_c": df_cuaca["suhu_rata2_c"].mean(),
    "total_curah_hujan_mm": df_cuaca["curah_hujan_mm"].sum(),
    "angin_maks_kmh": df_cuaca["kecepatan_angin_maks_kmh"].max()
}])
df_hasil = df_akomodasi.merge(df_cuaca_ringkas, on="kecamatan", how="left", validate="many_to_one", indicator=True)
jumlah_cocok = int((df_hasil["_merge"] == "both").sum())
jumlah_tidak_cocok = int((df_hasil["_merge"] != "both").sum())
df_hasil = df_hasil.drop(columns=["_merge"])
if len(df_hasil) != len(df_akomodasi) or jumlah_tidak_cocok:
    raise ValueError("Validasi merge gagal.")
if df_hasil.duplicated().any():
    raise ValueError("Ditemukan duplikasi pada hasil merge.")

OUTPUT_MERGE = PROCESSED_DIR / "akomodasi_cuaca_ubud.csv"
df_hasil.to_csv(OUTPUT_MERGE, index=False, encoding="utf-8-sig")
print("Baris sumber:", len(df_akomodasi))
print("Baris hasil:", len(df_hasil))
print("Baris cocok:", jumlah_cocok)
print("Baris tidak cocok:", jumlah_tidak_cocok)
print("Jumlah kolom hasil:", len(df_hasil.columns))
display(df_hasil.head())


## 11. Pengukuran ukuran penyimpanan aktual

In [ ]:
# Ukur ukuran aktual seluruh file proyek yang sudah terbentuk sampai tahap pengolahan.
# File pengukuran dan provenance dikecualikan agar tidak terjadi pengukuran diri/siklus.
ukuran_rows = []
for path_file in sorted(BASE_DIR.rglob("*")):
    if not path_file.is_file():
        continue
    relative = path_file.relative_to(BASE_DIR).as_posix()
    if relative.startswith(".git/") or "__pycache__" in path_file.parts:
        continue
    if relative in {
        "metadata/storage_measurement.csv",
        "metadata/provenance.json",
        "metadata/provenance.csv",
    }:
        continue

    byte = path_file.stat().st_size
    if relative.startswith("data/raw/"):
        kategori = "data mentah"
    elif relative.startswith("data/interim/"):
        kategori = "data sementara/hasil ekstraksi"
    elif relative.startswith("data/processed/"):
        kategori = "data hasil pengolahan"
    elif relative.startswith("metadata/"):
        kategori = "metadata"
    elif relative.startswith("docs/"):
        kategori = "dokumentasi"
    else:
        kategori = "file proyek lainnya"

    ukuran_rows.append({
        "nama_file": path_file.name,
        "path": relative,
        "kategori": kategori,
        "ekstensi": path_file.suffix.lower() or "(tanpa ekstensi)",
        "ukuran_byte": byte,
        "ukuran_kb": round(byte / 1024, 3),
        "ukuran_mb": round(byte / (1024 ** 2), 6),
    })

df_ukuran = pd.DataFrame(ukuran_rows)
if df_ukuran.empty:
    raise ValueError("Tidak ada file proyek yang dapat diukur.")

df_ukuran = df_ukuran.sort_values("ukuran_byte", ascending=False).reset_index(drop=True)
STORAGE_PATH = META_DIR / "storage_measurement.csv"
df_ukuran.to_csv(STORAGE_PATH, index=False, encoding="utf-8-sig")

total_byte = int(df_ukuran["ukuran_byte"].sum())
print("Jumlah file yang diukur:", len(df_ukuran))
print("Total ukuran aktual (byte):", total_byte)
print("Total ukuran aktual (KB):", round(total_byte / 1024, 3))
print("Total ukuran aktual (MB):", round(total_byte / (1024 ** 2), 6))
print("File pengukuran:", STORAGE_PATH)
display(df_ukuran)


## 12. Provenance dan hash MD5

In [ ]:
def md5_file(path_file):
    h = hashlib.md5()
    with open(path_file, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

file_sumber = [
    (PDF_PATH, "BPS Kabupaten Gianyar", "PDF", "https://gianyarkab.bps.go.id/", "Publikasi Direktori Hotel Kabupaten Gianyar 2017; direktori tahun 2015."),
    (DATASET_PATH, "Ekstraksi publikasi BPS", "CSV", "Dokumentasi ekstraksi dari PDF BPS", "Dataset akomodasi Ubud hasil ekstraksi."),
    (WIKI_RAW_PATH, "Wikipedia bahasa Indonesia", "JSON", WIKI_API, "Respons mentah pencarian MediaWiki API."),
    (WIKI_CSV_PATH, "Wikipedia bahasa Indonesia", "CSV", WIKI_API, "Metadata artikel hasil ekstraksi dari respons mentah."),
    (API_RAW_PATH, "Open-Meteo Historical Weather API", "JSON", API_URL, "Respons mentah cuaca harian Januari 2025."),
    (CUACA_CSV_PATH, "Open-Meteo Historical Weather API", "CSV", API_URL, "Data cuaca harian hasil ekstraksi JSON."),
    (OSM_RAW_PATH, "OpenStreetMap contributors melalui Overpass API", "JSON", OVERPASS_URL, "Respons mentah objek akomodasi pada bounding box perkiraan Ubud; lisensi ODbL."),
    (OSM_CSV_PATH, "OpenStreetMap contributors melalui Overpass API", "CSV", OVERPASS_URL, "Ekstraksi objek akomodasi dari respons Overpass; lisensi ODbL."),
    (OSM_ERROR_PATH, "Overpass API", "TXT", OVERPASS_URL, "Catatan kegagalan akuisisi jika endpoint tidak dapat diakses."),
    (OUTPUT_MERGE, "Pengolahan notebook", "CSV", "Dataset BPS + ringkasan cuaca", "Hasil penggabungan data akomodasi dan konteks cuaca."),
    (INTERIM_DIR / "analisis_akomodasi_per_desa.csv", "Pengolahan notebook", "CSV", "Dataset BPS", "Ringkasan jumlah akomodasi per desa."),
    (INTERIM_DIR / "analisis_akomodasi_per_klasifikasi.csv", "Pengolahan notebook", "CSV", "Dataset BPS", "Ringkasan jumlah akomodasi per klasifikasi."),
    (STORAGE_PATH, "Pengukuran lokal", "CSV", "Pengukuran filesystem", "Ukuran file aktual dalam byte, KB, dan MB."),
]

provenance_records = []
for path_file, sumber, format_file, url_sumber, keterangan in file_sumber:
    if path_file.exists() and path_file.is_file():
        provenance_records.append({
            "nama_file": path_file.name,
            "path": path_file.relative_to(BASE_DIR).as_posix(),
            "sumber": sumber,
            "format": format_file,
            "url_sumber": url_sumber,
            "ukuran_byte": path_file.stat().st_size,
            "md5": md5_file(path_file),
            "waktu_pencatatan_utc": datetime.now(timezone.utc).isoformat(),
            "keterangan": keterangan,
        })

df_provenance = pd.DataFrame(provenance_records)
PROVENANCE_JSON = META_DIR / "provenance.json"
PROVENANCE_CSV = META_DIR / "provenance.csv"
PROVENANCE_JSON.write_text(
    json.dumps(provenance_records, ensure_ascii=False, indent=2),
    encoding="utf-8"
)
df_provenance.to_csv(PROVENANCE_CSV, index=False, encoding="utf-8-sig")

print("Jumlah file tercatat dalam provenance:", len(df_provenance))
display(df_provenance)


## 13. Kesimpulan

In [ ]:
print("RINGKASAN UTS DAM")
print("=" * 45)
print(f"Dataset akomodasi BPS: {len(df_akomodasi)} baris, {len(df_akomodasi.columns)} kolom")
print(f"Data cuaca Open-Meteo: {len(df_cuaca)} observasi harian")
print(f"Status pemeriksaan robots.txt BPS: {status_scraping}")
print(f"Objek akomodasi OpenStreetMap: {len(df_osm)}")
print(f"Hasil penggabungan: {len(df_hasil)} baris, {len(df_hasil.columns)} kolom")
print(f"Baris cocok: {jumlah_cocok}; tidak cocok: {jumlah_tidak_cocok}")
print(f"Duplikasi hasil: {df_hasil.duplicated().sum()}")
print(f"Ukuran proyek terukur: {round(total_byte / (1024 ** 2), 6)} MB")
print("File JSON mentah API dan provenance dibuat oleh notebook jika seluruh sel berhasil.")
print("Perhatian: data BPS tahun 2015 dan cuaca Januari 2025 berbeda periode.")
print("Data OpenStreetMap merupakan data tambahan dalam bounding box perkiraan, bukan direktori resmi atau scraping HTML.")
